## Clean up masks

In [ ]:
from pathlib import Path

from bioio import BioImage
import bioio_ome_tiff
import bioio_tifffile
from bioio.writers import OmeTiffWriter

import numpy as np
import pandas as pd

%matplotlib notebook
%matplotlib inline
import matplotlib.pyplot as plt

import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn
from src.d01_init_proc import vis_and_rescale
from src.d01_init_proc import subtractbg

from scipy import ndimage as ndi
from skimage.measure import label, regionprops

from skimage.filters import threshold_otsu, threshold_multiotsu
from skimage import morphology
import numpy.ma as ma

%load_ext autoreload
#%autoreload 2

In [ ]:
input_dirpath = Path(input())

In [ ]:
df = pd.read_csv(input_dirpath)
df.head()

In [ ]:
df = df[df['notes'] == 'clean up']

In [ ]:
bin_dirpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/bg_subtracted/masked_cell_caax_binch1_init')
sel_dirpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/bg_subtracted/masked_cell_caax_binch1_sel_copy')

In [ ]:
idc = df.index
idc

In [ ]:
idx = 23

imgname = df.at[idx, 'input image name'].split('.ome.tif')[0]
print(imgname)

In [ ]:
selected_param_shape = 4
selected_param_holes = 5

In [ ]:
orig_img_ch = 0
bgsb_img_ch = 1

imgname_shape = f'{imgname}_p{selected_param_shape}.ome.tif'
shape_imgpath = bin_dirpath / imgname_shape

img_file = BioImage(shape_imgpath, reader=bioio_ome_tiff.Reader)
img = img_file.data

orig_img = img[:, orig_img_ch, np.newaxis, :, :, :]
orig_img, _ = vis_and_rescale.rescale_img(orig_img, target_perc_grayval=30)

bin_shape = (img[:, bgsb_img_ch, np.newaxis, :, :, :] > 0).astype('bool')

fig_imgs = [orig_img, bin_shape]
fig_labels = ['orig img', 'binary shape']

if selected_param_holes is not None:
    imgname_holes = f'{imgname}_p{selected_param_holes}.ome.tif'
    holes_imgpath = bin_dirpath / imgname_holes
    bin_holes = BioImage(holes_imgpath, reader=bioio_ome_tiff.Reader).data
    bin_holes = (bin_holes[:, bgsb_img_ch, np.newaxis, :, :, :] > 0).astype('bool')
    fig_imgs.append(bin_holes)
    fig_labels.append('binary holes')

fig = vis_and_rescale.create_fig(fig_imgs, fig_labels)

In [ ]:
def edit_bin_mask(bin_img, fig_imgs=None):
    bin_img = bin_img.astype('bool')
    size_t = bin_img.shape[0]
    for t in range(size_t):
        bin_img[t, :, :, :, :] = morphology.remove_small_objects(bin_img[t, :, :, :, :], 100)
        bin_img[t, :, :, :, :] = morphology.remove_small_holes(bin_img[t, :, :, :, :], 100)

    if fig_imgs is not None:
        fig_imgs.append(bin_img)
    return bin_img, fig_imgs

# fig_imgs = []
# bin_shape_edited, fig_imgs = edit_bin_mask(bin_shape, fig_imgs)

if selected_param_holes is not None:
    bin_holes_edited, fig_imgs = edit_bin_mask(bin_holes, fig_imgs)

if len(fig_imgs) > 0:
    fig = vis_and_rescale.create_fig(fig_imgs)

In [ ]:
if selected_param_holes is not None:
    print(imgname_holes)
    edited_holes = img
    edited_holes[:, bgsb_img_ch, np.newaxis, :, :, :] = (bin_holes_edited.astype('int') * np.iinfo(img_file.dtype).max).astype(img_file.dtype)
    
    ome_metadata = utils.construct_ome_metadata(edited_holes, img_file)
    OmeTiffWriter.save(edited_holes, sel_dirpath / f'holes_{imgname_holes}', ome_xml=ome_metadata)

In [ ]:
bin_edited = bin_shape_edited

maskpath = input()
if not maskpath =='NA':
    
    mask = BioImage(Path(maskpath), reader=bioio_tifffile.Reader).data

    bin_holes_masked = (bin_holes_edited > 0).astype('int') * (mask > 0).astype('int')
    mask = np.broadcast_to(mask, bin_shape_edited.shape)

    bin_edited[mask > 0] = bin_holes_masked[mask > 0]
    fig = vis_and_rescale.create_fig([bin_edited])

In [ ]:
bin_edited_stack = img
bin_edited_stack[:, bgsb_img_ch, np.newaxis, :, :, :] = (bin_edited.astype('int') * np.iinfo(img_file.dtype).max).astype(img_file.dtype)

ome_metadata = utils.construct_ome_metadata(bin_edited_stack, img_file)
OmeTiffWriter.save(bin_edited_stack, sel_dirpath / imgname_shape, ome_xml=ome_metadata)
print(imgname_shape)

In [ ]:
# Optional: resave Tiff edited in ImageJ as OmeTiff
# bin_edited_stack = BioImage(sel_dirpath / imgname_shape, reader=bioio_tifffile.Reader).data
# OmeTiffWriter.save(bin_edited_stack, sel_dirpath / imgname_shape, ome_xml=ome_metadata)